In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm

## Question 1
Suppose that a process can be modeled via linear regression: 

W = np.random.normal(0, 1, (1000,))  
X = W + np.random.normal(0, 1, (1000,))   
Z = np.random.normal(0, 1, (1000,))   
Y = X + Z + W + np.random.normal(0, 1, (1000,))

Which is closest to the correlation of X with the error term in the equation for Y? 

- A: 0

- B: -0.50

- C: 1.0

- D: 0.5

- E: 0.75

In [2]:
W = np.random.normal(0, 1, (1000,))  
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,))   
Y = X + Z + W + np.random.normal(0, 1, (1000,))

Answer: A   
____

## Question 2

If Y is written as depending on X and Z only, W is part of the error term. Which, then, is closest to the expected correlation of X with the error term in the equation for Y? 

- A: 0

- B: 0.50

- C: 1.0

- D: 0.75

- E: -0.50

In [6]:
np.corrcoef(Y-X-Z, X)[0, 1]

np.float64(0.503076327287177)

Answer: B   
____

## Question 3

In the data frame for homework_7.1.csv, control for W by regressing Y on X and Z at the following constant values of W: -1, 0, and 1.  
(You cannot literally use a constant value of W, of course, or you will have only one data point! How will you manage this?) The question is: Is the coefficient of X  

- A: decreasing

- B: staying about the same (say, within 0.2 or so) as W increases? 

- C: increasing

In [7]:
df= pd.read_csv("../data/homework_7.1.csv")
df.head()

,X,W,Z,Y
0,1.137055,1.221768,0.327829,1.944532
1,-0.112905,0.465835,0.599650,0.655514
2,2.077755,1.795414,-0.063393,5.934411
3,0.456373,-0.512159,1.177413,-0.188064
4,-1.012402,0.080002,-0.275697,-0.533775


In [8]:
model = sm.OLS(df['Y'], sm.add_constant(df[['X', "W",'Z']])).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       0.840
Model:                            OLS   Adj. R-squared:                  0.840
Method:                 Least Squares   F-statistic:                 1.745e+04
Date:                Sat, 17 Oct 2026   Prob (F-statistic):               0.00
Time:                        17:29:15   Log-Likelihood:                -17112.
No. Observations:               10000   AIC:                         3.423e+04
Df Residuals:                    9996   BIC:                         3.426e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.5184      0.013     38.686      0.000       0.492       0.545
X              1.4096      0.013    105.035      0.000       1.383       1.436
W              1.2034      0.019     63.133      0.000       1.166       1.241
Z             -0.5989      0.014    -44.063      0.000      -0.626      -0.572
==============================================================================
Omnibus:                     1420.417   Durbin-Watson:                   1.982
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             3489.559
Skew:                           0.810   Prob(JB):                         0.00
Kurtosis:                       5.398   Cond. No.                         2.67
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [11]:
for w in [-1, 0, 1]:

    subset = df[(df["W"] >= w - 0.3) &(df["W"] <= w + 0.3)]

    X = sm.add_constant(subset[["X", "Z"]])
    y = subset["Y"]

    model = sm.OLS(y, X).fit()

    print("W:", w)
    print("Number of observations:", len(subset))
    print("Coefficient of X:", model.params["X"])
    print()

W: -1
Number of observations: 1463
Coefficient of X: 0.9175864258499985

W: 0
Number of observations: 2339
Coefficient of X: 1.4282475794761718

W: 1
Number of observations: 1396
Coefficient of X: 1.9314102613690554



Answer: C   
____

## Question 4

In [13]:
def make_error(corr_const, num): 

    err = list() 
    prev = np.random.normal(0, 1) 

    for n in range(num): 

        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
        err.append(prev) 

    return np.array(err) 

Create a linear regression model that uses this function as the error for both (a) the treatment, X, and (b) the outcome, Y. (You can use random normal error for any other covariates, if you have them.)    

As corr_const increases from 0.2 to 0.5 to 0.8, find (i) the standard deviation of the estimate of the X coefficient over many trials, and (ii) the mean of the standard error estimate of the X coefficient over many trials.   

When corr_const increases, then:  

Hint: don't forget to include an intercept in your regression

- A: (i) and (ii) remain about the same

- B: The ratio (i) / (ii) increases

- C: The ratio (i) / (ii) decreases

- D: (i) and (ii) differ, but their ratio remains about the same

In [14]:
n = 300
trials = 1000

for corr_const in [0.2, 0.5, 0.8]:

    coefficients = []
    standard_errors = []

    for _ in range(trials):

        # Generate data
        x = 5 + make_error(corr_const, n)
        error_y = make_error(corr_const, n)
        y = 3 + 2 * x + error_y

        # Fit OLS
        model = sm.OLS(y, sm.add_constant(x)).fit()

        # Save coefficient and standard error
        coefficients.append(model.params[1])
        standard_errors.append(model.bse[1])

    # (i) Standard deviation of estimated coefficients
    i = np.std(coefficients, ddof=1)

    # (ii) Mean of reported OLS standard errors
    ii = np.mean(standard_errors)

    # Ratio
    ratio = i / ii

    print(f"Correlation: {corr_const}")
    print(f"SD of coefficients: {i:.4f}")
    print(f"Mean OLS SE: {ii:.4f}")
    print(f"Ratio (i)/(ii): {ratio:.3f}")
    print("-" * 35)

Correlation: 0.2
SD of coefficients: 0.0602
Mean OLS SE: 0.0580
Ratio (i)/(ii): 1.038
-----------------------------------
Correlation: 0.5
SD of coefficients: 0.0723
Mean OLS SE: 0.0578
Ratio (i)/(ii): 1.251
-----------------------------------
Correlation: 0.8
SD of coefficients: 0.1265
Mean OLS SE: 0.0576
Ratio (i)/(ii): 2.198
-----------------------------------


Answer: B   
_____